**📌 Step 1 : Load the CSV file into a DataFrame**

In [ ]:
import pandas as pd
import re
from data_cleaning import load_commented_rows, build_cleaned_comments

# Load the comments of all daily snapshots of Snapshot_2026 (rows with a comment only)
raw = load_commented_rows()
# Clean the comments (remove the "pseudo, YYYY-MM-DD: " prefix) and keep one row per
# (material, vendor, comment) with First_Seen / Last_Seen / N_Snapshots / Days_Active
df = build_cleaned_comments(raw)

print("Colonnes :", df.columns.tolist())
print("\nExemple de commentaires nettoyés :")
print(df["Comment"].head())

**📌 Step 2 : Data Cleaning**

In [ ]:
# Cleaning (prefix, empty comments, duplicates) is done in build_cleaned_comments
print(f"Nombre de commentaires uniques : {len(df)}")

In [ ]:
# %% [Affichage des commentaires après nettoyage]

# Afficher un échantillon de commentaires
print("\n=== ÉCHANTILLON DE COMMENTAIRES ===")
print(df["Comment"].head(10))

# Afficher les commentaires avec leur contexte
print("\n=== COMMENTAIRES AVEC CONTEXTE ===")
context_df = df[["Comment", "Root_Cause", "VENDOR_NAME", "MATERIAL_NUMBER"]]
print(context_df.head(10))

# Afficher les commentaires les plus fréquents
from collections import Counter
comment_counts = Counter(df["Comment"])
print("\n=== COMMENTAIRES LES PLUS FRÉQUENTS ===")
for comment, count in comment_counts.most_common(10):
    print(f"{count}x: {comment}")

# Export (the column list is USEFUL_COLUMNS in data_cleaning.py)
# Then run translation.py (or translation.ipynb) to add Comment_EN
useful_columns_doc = [
    'Snapshot_Date', 'MATERIAL_NUMBER', 'PLANT', 'MRP_CONTROLLER',
    'MATERIAL_DESC', 'RESCHEDULING_DATE', 'VENDOR_ACCOUNT_NUMBER',
    'VENDOR_NAME', 'ITEM_DELIVERY_DATE', 'PO_QTY', 'INVENTORY_QTY',
    'STOCKOUT_DATE', 'BLOCKED_STOCK', 'QUALITY_INSPECTION_STOCK',
    'ORDER_CONFIRMATION', 'PURCHASING_GROUP', 'SOURCING_MANAGER',
    'Delay', 'Horizon', 'Total', 'MATERIAL_STATUS', 'MATERIAL_RISK',
    'Comment', 'Root_Cause', 'Impact', 'Instruments_Affected'
]

df.to_csv("../data/cleaned_comments.csv", index=False)
print("\n✅ Commentaires nettoyés exportés dans data/cleaned_comments.csv")

**📌 Étape 3 : Tokenisation & Lemmatisation**


In [ ]:
import spacy

# Charger les modèles spaCy pour l'anglais et l'allemand
nlp_en = spacy.load("en_core_web_sm")
nlp_de = spacy.load("de_core_news_sm")

# Fonction pour détecter la langue (optionnel)
from langdetect import detect

def detect_language(text):
    try:
        return detect(text)
    except:
        return "en"  # Par défaut en anglais

# Fonction pour nettoyer, tokeniser et lemmatiser
def clean_tokenize_lemmatize(text, lang="en"):
    if pd.isna(text):
        return []

    # Nettoyage de base
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Supprime les caractères spéciaux

    # Tokenisation et lemmatisation
    if lang == "en":
        doc = nlp_en(text)
    else:
        doc = nlp_de(text)

    # Retourne les lemmes (racines des mots)
    return [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

# Appliquer à la colonne Comment
df["Tokens"] = df["Comment"].apply(
    lambda x: clean_tokenize_lemmatize(x, lang=detect_language(x)) if pd.notnull(x) else []
)

# Afficher un exemple
print("\nExemple de tokens/lemmes :")
print(df["Tokens"].iloc[0])

📌 **Step 4 : Analyse Key-words by Root-Causes**

In [ ]:
from collections import Counter

# 1. Extraire tous les tokens dans une liste
all_tokens = [token for tokens in df["Tokens"] for token in tokens]
token_counts = Counter(all_tokens)

# 2. Afficher les 20 mots-clés les plus fréquents
print("\nTop 20 mots-clés :")
print(token_counts.most_common(20))

# 3. (Optionnel) Afficher les mots-clés par root cause
root_cause_keywords = {}
for root_cause in df["Root_Cause"].unique():
    subset = df[df["Root_Cause"] == root_cause]
    tokens = [token for tokens in subset["Tokens"] for token in tokens]
    root_cause_keywords[root_cause] = Counter(tokens).most_common(10)

for root_cause, keywords in root_cause_keywords.items():
    print(f"\nTop mots-clés pour '{root_cause}' :")
    print(keywords)

**📌 Step 5 : Visualisations**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Compter les root causes par fournisseur
supplier_root_cause = df.groupby(["VENDOR_NAME", "Root_Cause"]).size().unstack(fill_value=0)

# Afficher la heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(supplier_root_cause, cmap="YlOrRd", annot=True, fmt="d")
plt.title("Root Causes par Fournisseur")
plt.xlabel("Root Cause")
plt.ylabel("VENDOR_NAME")
plt.show()

Step 6 : Test 

In [ ]:
import sys
sys.path.append("..")
import importlib
import stats_engine
importlib.reload(stats_engine)
from stats_engine import compute_stats

import pandas as pd

df = pd.read_csv("../data/cleaned_comments.csv")
stats = compute_stats(df)
print(stats["first_snapshot"], "→", stats["last_snapshot"])
print(stats["top_root_causes"])

In [ ]:
print(stats["first_snapshot"], "→", stats["last_snapshot"])

In [ ]:
stats["top_root_causes"]